# Dapuria — Production Retrieval & Reranking Pipeline

Notebook ini adalah versi implementasi final setelah eksperimen dan evaluasi dibekukan.

**Konfigurasi final:**
- SBERT: `LazarusNLP/all-indo-e5-small-v4`
- ChromaDB collection: `resep_masakan`
- Semantic retrieval: Top-50
- Keyword retrieval tambahan: maksimal 20 kandidat non-duplikat
- Similarity untuk WLC: min-max normalization per candidate pool
- Efficiency: harmonic mean `Coverage_recipe` dan `Coverage_user`
- WLC final: `0.7 × semantic + 0.3 × efficiency`
- Guardrail rule-based: `T_jumlah=1`, `T_kategori`, `T_relevansi=min(2, |U|)`
- Llama 3 hanya untuk penyampaian/clarification; tidak digunakan untuk parsing, retrieval, reranking, atau keputusan threshold.

Notebook ini tidak memuat grid search atau evaluasi test. Semua parameter dianggap **frozen**.


## 1. Import production module\n

In [1]:
from dapuria_pipeline import (
    DapuriaRecipeService,
    MODEL_NAME,
    TOP_K_SBERT,
    MAX_KEYWORD_EXTRA,
    W1_FINAL,
    W2_FINAL,
    parse_user_ingredients,
    ingredient_matches,
    coverage_stats,
    harmonic_efficiency,
)

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Inisialisasi model dan ChromaDB\n

In [2]:
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

service.healthcheck()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11706.40it/s]


{'model_name': 'LazarusNLP/all-indo-e5-small-v4',
 'collection_name': 'resep_masakan',
 'document_count': 5000,
 'config': {'top_k_sbert': 50, 'max_keyword_extra': 20, 'w1': 0.7, 'w2': 0.3}}

## 3. Sanity checks\n

In [3]:
# Sanity checks untuk bug yang sebelumnya ditemukan
assert ingredient_matches("teri", "terigu") is False
assert ingredient_matches("ayam", "dada ayam") is True

assert parse_user_ingredients(
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
) == ["telur", "dada ayam", "bawang putih"]

print("Sanity checks: OK")

Sanity checks: OK


## 4. Contoh rekomendasi end-to-end\n

In [4]:
# Contoh request production
user_text = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

result = service.handle_user_input(
    user_text,
    top_n=5,
)

print("status:", result["status"])
print("parsed:", result.get("user_ingredients"))

if result["status"] == "ok":
    for rank, rec in enumerate(result["results"], start=1):
        print(
            f"[{rank}] {rec['title']} | "
            f"final={rec['final_score']:.4f} | "
            f"semantic={rec['sbert_score_norm']:.4f} | "
            f"eff={rec['skor_efisiensi_bahan']:.4f}"
        )
        print("  bahan user terpakai:", rec["matched_user_items"])
        print("  bahan resep cocok  :", rec["matched_recipe_items"])
        print("  bahan perlu tambah :", rec["missing_items"])

status: no_match
parsed: ['telur', 'dada ayam', 'bawang putih']


## 5. Guardrail smoke test\n

In [5]:
# Smoke test guardrail
tests = [
    "",
    "garam, gula, kecap manis",
    "soda kue, kunyit",
    "telur, tahu",
]

for q in tests:
    result = service.handle_user_input(q, top_n=5)
    print("=" * 70)
    print("INPUT :", repr(q))
    print("STATUS:", result["status"])
    if result["status"] != "ok":
        print("RESP  :", result["template"])
    else:
        print("TOP-1 :", result["results"][0]["title"])

INPUT : ''
STATUS: no_ingredients
RESP  : Sepertinya belum ada bahan yang terdeteksi dari pesanmu. Boleh sebutkan bahan yang kamu punya?
INPUT : 'garam, gula, kecap manis'
STATUS: need_main_ingredient
RESP  : Bahan yang kamu sebutkan (garam, gula, kecap manis) belum memuat bahan utama yang dikenali. Tambahkan minimal satu bahan utama, misalnya ayam, telur, tahu, tempe, ikan, nasi, kentang, atau sayuran.
INPUT : 'soda kue, kunyit'
STATUS: need_main_ingredient
RESP  : Bahan yang kamu sebutkan (soda kue, kunyit) belum memuat bahan utama yang dikenali. Tambahkan minimal satu bahan utama, misalnya ayam, telur, tahu, tempe, ikan, nasi, kentang, atau sayuran.
INPUT : 'telur, tahu'
STATUS: ok
TOP-1 : Telur Dadar Tahu (Tebal, Gurih, Padat)


## 6. Payload untuk Llama 3 / NLG\n

In [6]:
# Payload aman untuk diberikan ke Llama 3 / layer NLG.
# Llama 3 hanya boleh memverbalisasi data ini, tidak membuat resep baru.
pipeline_result = service.handle_user_input(
    "telur, tahu",
    top_n=5,
)

nlg_payload = service.build_nlg_payload(pipeline_result)
nlg_payload

{'status': 'ok',
 'instruction': 'Gunakan hanya data resep berikut. Jangan menambah bahan, langkah, nama resep, atau fakta yang tidak ada pada data.',
 'user_ingredients': ['telur', 'tahu'],
 'recipes': [{'recipe_id': 'resep_1038',
   'title': 'Telur Dadar Tahu (Tebal, Gurih, Padat)',
   'ingredients': ['tahu kuning (agar makin padat',
    'telur ayam',
    'garam',
    'msg',
    'lada putih',
    'minyak goreng'],
   'steps': '1. Siapkan bahan, hancurkan tahu menggunakan garpu 2. Masukkan 4 butir telur. Kocok lepas dan tambahkan bumbu seasoning 3. Goreng di minyak yang sudah dipanaskan. Tunggu hingga sisi bawahnya matang lalu balik. Kemudian tunggu hingga matang sempurna 4. Angkat dan sajikan',
   'matched_user_items': ['tahu', 'telur'],
   'missing_items': ['garam', 'msg', 'lada putih', 'minyak goreng']},
  {'recipe_id': 'resep_1389',
   'title': 'Dadar Tahu Telur',
   'ingredients': ['telur',
    'tahu putih',
    'daun bawang',
    'merica bubuk',
    'masako',
    'garam',
    'm

## 7. Wrapper untuk UI/backend\n

In [7]:
# Contoh fungsi wrapper yang bisa dipanggil UI/backend.
def recommend_recipes(user_text: str, top_n: int = 5) -> dict:
    return service.handle_user_input(
        user_text=user_text,
        top_n=top_n,
    )


# contoh
response = recommend_recipes("ayam, kentang, tomat")
response["status"]

'ok'